# Artmatcher

This notebook demonstrates the artmatcher module for image matching and artwork recognition.

The artmatcher package provides:
- **Image Stitching**: Combine multiple overlapping images into a panorama using SIFT features
- **Artwork Recognition**: Build a SIFT feature database and find matching paintings
- **Similarity Classification**: Determine if a top similarity score represents a true match

Main technologies used:
- [OpenCV SIFT](https://docs.opencv.org/4.x/da/df5/tutorial_py_sift_intro.html) - Scale-Invariant Feature Transform for keypoint detection
- [FLANN Matcher](https://docs.opencv.org/4.x/d5/d6f/tutorial_feature_flann_matcher.html) - Fast Library for Approximate Nearest Neighbors
- [Homography](https://docs.opencv.org/4.x/d9/dab/tutorial_homography.html) - Geometric transformation between image planes
- [ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html) - Oriented FAST and Rotated BRIEF (fast alternative to SIFT)
- [Perceptual Hashing](https://github.com/JohannesBuchner/imagehash) - Image similarity using perceptual hashes

# Setup

### Dependencies

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from artmatcher import (
    MatchResult,
    build_database,
    find_painting,
)

### Configs

**Note:** You need to modify these paths to point to your image directories.

In [ ]:
# Path to reference paintings (e.g., WikiArt dataset)
PAINTINGS_DIR = "/path/to/paintings"  # MODIFY ME

# Path to query images (e.g., photos of paintings)
QUERIES_DIR = "/path/to/queries"  # MODIFY ME

# Path to save the recognition database
DATABASE_PATH = Path("./artmatcher_database.pkl")

---

# Part 1: Artwork Recognition

Artwork recognition finds which painting(s) a query image matches in a database. The process involves:

1. **Building a Database**: Extract and store SIFT features from all reference paintings
2. **Query Matching**: Extract features from query image and match against database
3. **Geometric Verification**: Use homography to verify the match is geometrically consistent

```mermaid
flowchart TD
  subgraph Database Building
    paintings[Paintings] -->|SIFT| db[(Feature Database)]
  end
  subgraph Query
    query[Query Image] -->|SIFT| qf[Query Features]
    qf -->|FLANN| match[Match Candidates]
    match -->|Homography| verify[Verified Matches]
  end
  db --> match
```

## 1.1 Build the Recognition Database

First, we extract SIFT features from all reference paintings and store them in a database.

In [ ]:
# Get all painting images
painting_paths = sorted(glob.glob(f"{PAINTINGS_DIR}/*.jpg"))
print(f"Found {len(painting_paths)} paintings")

# Limit for demo (remove for full dataset)
painting_paths = painting_paths[:100]
print(f"Using {len(painting_paths)} paintings for demo")

In [ ]:
# Build the database (this may take a while for large datasets)
if painting_paths:
    database = build_database(
        painting_paths,
        workers=4,  # Number of parallel workers
        max_size=800,  # Resize images before feature extraction (for speed)
        max_features=1000,  # Limit features per image (500-2000 recommended)
        on_progress=print,
    )

    print(f"\nDatabase contains {len(database.features)} paintings")

    # Save for later use
    database.save(DATABASE_PATH)
    print(f"Database saved to {DATABASE_PATH}")
else:
    print("No paintings found. Update PAINTINGS_DIR path.")

## 1.2 Load an Existing Database

If you've already built a database, you can load it from disk.

In [ ]:
# Load existing database (uncomment if you have one saved)
# database = RecognitionDatabase.load(DATABASE_PATH)
# print(f"Loaded database with {len(database.features)} paintings")

## 1.3 Find Matching Paintings

Query the database to find which painting(s) match a given image.

In [ ]:
# Query image (e.g., a photo of a painting)
query_path = ""  # MODIFY ME: path to your query image

if query_path and "database" in dir():
    results = find_painting(
        query_path,
        database,
        min_matches=10,  # Minimum good matches to consider
        min_inliers=4,  # Minimum inliers after homography
        max_results=5,  # Number of top results to return
        verify_geometry=True,  # Use homography verification (recommended)
    )

    print(f"Found {len(results)} potential matches:\n")
    for i, match in enumerate(results, 1):
        print(f"{i}. {match.painting_name}")
        print(f"   Matches: {match.num_matches}, Inliers: {match.num_inliers}")
        print(f"   Confidence: {match.confidence:.4f}")
        print()
else:
    print("Set query_path and ensure database is built")

## 1.4 Visualize Match Results

Display the query image alongside the best match.

In [ ]:
def show_match(query_path: str, match: MatchResult):
    """Display query and matched painting side by side."""
    _fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Query image
    query_img = Image.open(query_path)
    axes[0].imshow(query_img)
    axes[0].set_title("Query Image")
    axes[0].axis("off")

    # Matched painting
    match_img = Image.open(match.painting_path)
    axes[1].imshow(match_img)
    axes[1].set_title(
        f"Match: {match.painting_name}\nConfidence: {match.confidence:.4f}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


# Show the best match
if query_path and "results" in dir() and results:
    show_match(query_path, results[0])
else:
    print("Run the query cell above first")

---

# Part 2: ML Classifier

The `MLClassifier` provides robust classification by combining:

- **Distribution features**: Statistics from the similarity distribution (gap, z-score, decay rate)
- **ORB features**: Geometric verification using ORB keypoints and homography
- **Perceptual hashes**: Multiple hash types (pHash, dHash, wHash, aHash)

These features are fed into a calibrated Random Forest classifier trained on labeled examples.

In [ ]:
from artmatcher.ml_classifier import FeatureExtractor, MLClassifier

# List of features used by the ML classifier
print("ML Classifier Feature Order:")
for i, feature in enumerate(MLClassifier.FEATURE_ORDER, 1):
    print(f"  {i:2d}. {feature}")

In [ ]:
# Example: Extract features from similarity scores
extractor = FeatureExtractor()

# Distribution features only (fast, no images needed)
features = extractor.extract_distribution_features([0.8, 0.5, 0.4, 0.4, 0.3])

print("Distribution features:")
for name, value in features.items():
    print(f"  {name}: {value:.4f}")

In [ ]:
# Example: Extract all features including ORB and hash (requires image paths)
# Uncomment and modify paths to run

# query_img_path = "/path/to/query.jpg"
# reference_img_path = "/path/to/reference.jpg"
#
# all_features = extractor.extract_all_features(
#     example_similarities_match,
#     glashelder_path=query_img_path,
#     wikiart_path=reference_img_path,
# )
#
# print("All features (including ORB and hash):")
# for name, value in all_features.items():
#     print(f"  {name}: {value:.4f}")

## 2.1 Training the ML Classifier

To train a classifier, you need a labeled dataset in JSON format:

```json
{
  "positives": [
    {"glashelder_path": "...", "wikiart_path": "...", "similarities": [...]},
    ...
  ],
  "negatives": [...],
  "false_positives": [...],
  "false_negatives": [...]
}
```

In [ ]:
# Example: Train a classifier (uncomment to run)

# classifier = MLClassifier(threshold=0.5)
#
# metrics = classifier.train(
#     dataset_path="/path/to/labeling_dataset.json",
#     n_estimators=100,
#     cv_folds=5,
#     extract_image_features=True,  # Set False for faster training without images
#     workers=4,
# )
#
# print(f"\nCross-validation accuracy: {metrics.cv_accuracy:.3f} +/- {metrics.cv_std:.3f}")
# print(f"Samples: {metrics.n_samples} ({metrics.n_positives} pos, {metrics.n_negatives} neg)")
#
# # Save the trained model
# classifier.save("trained_classifier.pkl")

In [ ]:
# Example: Load and use a trained classifier (uncomment to run)

# classifier = MLClassifier.load("trained_classifier.pkl")
#
# result = classifier.classify(
#     similarities=example_similarities_match,
#     glashelder_path="/path/to/query.jpg",
#     wikiart_path="/path/to/reference.jpg",
# )
#
# print(f"Is match: {result.is_match}")
# print(f"Confidence: {result.confidence:.4f}")

---

# Summary

The artmatcher module provides three main capabilities:

1. **Artwork Recognition** (`build_database`, `find_painting`)
   - Builds SIFT feature database from reference paintings
   - Queries database with geometric verification
   - Returns confidence-ranked matches

2. **Similarity Classification** (`MLClassifier`)
   - Determines if top similarity represents a true match
   - Simple: polynomial fitting approach
   - ML: Random Forest with distribution + ORB + hash features